# 复现 Fig. 3：每个暗物质晕中的 PBH 两体直接捕获率

这个 Notebook 只完成论文 Fig. 3 的参数选择、数值调用和绘图。物理公式没有复制到这里：

- `twobodycapture.py`：Eq. (11)、速度平均与 Appendix A(14) 的每晕捕获率；
- `pbhmassfunction.py`：单色和对数正态 PBH 质量分布；
- 本 Notebook：选择 $z=0$、晕质量网格、两种浓度模型和两种质量函数，并生成图片。

这样后续修改绘图范围或颜色不会影响主体物理模块。

## 1. 导入依赖并定位项目目录

如果从项目根目录启动 Jupyter，当前目录就是项目目录；如果从 `notebooks` 目录启动，项目目录是它的上一级。下面的代码同时兼容这两种情况。

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "twobodycapture.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# 把项目根目录加入 Python 的模块搜索路径，随后才能导入主体 .py 文件。
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pbhmassfunction as pmf
import twobodycapture as capture

PROJECT_ROOT

WindowsPath('D:/pbh formation/pbh merge/2body channel and numerical simulation')

## 2. 设置 Fig. 3 的公共参数

论文 Fig. 3 固定在红移 $z=0$，横轴覆盖 $10^3$ 到 $10^{15} M_\odot$。这里使用 121 个对数均匀点，因此每个十倍质量区间有 10 个小间隔。

`f_pbh=1` 是论文直接捕获计算采用的总 PBH 丰度。图注提到晕质量的一半在单体、一半在双体中，但同时说明宽双体在直接捕获问题中等效为单体，因此不再额外乘以 $0.5^2$。

In [2]:
REDSHIFT = 0.0
F_PBH = 1.0
HALO_MASSES_MSUN = np.logspace(3.0, 15.0, 121)

HALO_MASSES_MSUN[[0, 30, 60, 90, 120]]

array([1.e+03, 1.e+06, 1.e+09, 1.e+12, 1.e+15])

## 3. 构造两种 PBH 质量积分测度

单色谱使用一个 $30 M_\odot$ 的质量节点和权重 1。对数正态谱使用 $\mu=\ln(30 M_\odot)$、$\sigma=0.6$，并通过 Gauss–Legendre 求积转成 96 个质量节点与 `psi(m) dm` 权重。

两种分布最终具有相同的接口，所以后面都调用同一个 Appendix A(14) 每晕率函数。

In [3]:
monochromatic_nodes, monochromatic_weights = (
    pmf.monochromatic_mass_distribution(mass_msun=30.0)
)

def figure3_lognormal_pdf(mass_msun):
    return pmf.lognormal_mass_pdf(
        mass_msun,
        median_mass_msun=30.0,
        sigma=0.6,
    )

lognormal_nodes, lognormal_weights = pmf.continuous_mass_quadrature(
    figure3_lognormal_pdf,
    point_count=96,
)

{
    "单色谱节点数": monochromatic_nodes.size,
    "单色谱权重和": monochromatic_weights.sum(),
    "对数正态谱节点数": lognormal_nodes.size,
    "对数正态谱权重和": lognormal_weights.sum(),
}

{'单色谱节点数': 1,
 '单色谱权重和': np.float64(1.0),
 '对数正态谱节点数': 96,
 '对数正态谱权重和': np.float64(1.0000000000000033)}

## 4. 计算四条捕获率曲线

两个晕浓度模型和两个 PBH 质量函数两两组合：

1. Ludlow16 + 单色谱；
2. Prada12 + 单色谱；
3. Ludlow16 + 对数正态谱；
4. Prada12 + 对数正态谱。

对每个晕质量，Notebook 调用主体模块中的 `capture_rate_per_halo_a14_per_year`。返回值单位是 $\mathrm{yr}^{-1}$。

In [4]:
curves = {}

for concentration_model in ("ludlow16", "prada12"):
    print(f"正在计算 {concentration_model} + monochromatic ...")
    curves[f"{concentration_model}_monochromatic"] = np.array([
        capture.capture_rate_per_halo_a14_per_year(
            halo_mass_msun,
            z=REDSHIFT,
            mass_nodes_msun=monochromatic_nodes,
            mass_probability_weights=monochromatic_weights,
            concentration_model=concentration_model,
            f_pbh=F_PBH,
        )
        for halo_mass_msun in HALO_MASSES_MSUN
    ])

    print(f"正在计算 {concentration_model} + lognormal ...")
    curves[f"{concentration_model}_lognormal"] = np.array([
        capture.capture_rate_per_halo_a14_per_year(
            halo_mass_msun,
            z=REDSHIFT,
            mass_nodes_msun=lognormal_nodes,
            mass_probability_weights=lognormal_weights,
            concentration_model=concentration_model,
            f_pbh=F_PBH,
        )
        for halo_mass_msun in HALO_MASSES_MSUN
    ])

print("四条曲线计算完成。")

正在计算 ludlow16 + monochromatic ...
正在计算 ludlow16 + lognormal ...
正在计算 prada12 + monochromatic ...
正在计算 prada12 + lognormal ...
四条曲线计算完成。


下面只展示五个晕质量锚点，便于阅读具体数量级。这不是额外计算，所有数值都来自上一个单元格。

In [5]:
anchor_indices = [0, 30, 60, 90, 120]
np.column_stack([
    HALO_MASSES_MSUN[anchor_indices],
    curves["ludlow16_monochromatic"][anchor_indices],
    curves["prada12_monochromatic"][anchor_indices],
    curves["ludlow16_lognormal"][anchor_indices],
    curves["prada12_lognormal"][anchor_indices],
])

array([[1.00000000e+03, 2.20220410e-15, 5.07687064e-15, 2.49988620e-15,
        5.76313470e-15],
       [1.00000000e+06, 3.60380114e-14, 4.57550202e-14, 4.09094359e-14,
        5.19399377e-14],
       [1.00000000e+09, 5.07051919e-13, 4.54897011e-13, 5.75592470e-13,
        5.16387542e-13],
       [1.00000000e+12, 5.32543435e-12, 4.72907964e-12, 6.04529791e-12,
        5.36833118e-12],
       [1.00000000e+15, 4.14212383e-11, 5.96466841e-11, 4.70203385e-11,
        6.77094020e-11]])

## 5. 绘制 Fig. 3

主图和右下角插图使用完全相同的数据。插图只是把 $10^3$–$10^6 M_\odot$ 的低质量端放大，并没有重新计算捕获率。

In [7]:
curve_styles = [
    ("ludlow16_monochromatic", "Ludlow16 + mono.", "tab:blue", "-"),
    ("prada12_monochromatic", "Prada12 + mono.", "tab:orange", "--"),
    ("ludlow16_lognormal", "Ludlow16 + log-normal", "tab:green", "-."),
    ("prada12_lognormal", "Prada12 + log-normal", "tab:red", ":"),
]

figure, axis = plt.subplots(figsize=(7.2, 5.2))

for curve_key, label, color, line_style in curve_styles:
    axis.plot(
        HALO_MASSES_MSUN,
        curves[curve_key],
        color=color,
        linestyle=line_style,
        linewidth=2.0,
        label=label,
    )

axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlim(1.0e3, 1.0e15)
axis.set_ylim(1.0e-15, 1.0e-10)
axis.set_xlabel(r"Halo Mass ($M_\odot$)")
axis.set_ylabel(r"$R_{\rm halo}\;(\mathrm{yr}^{-1})$")
axis.legend(loc="upper left", fontsize=8.5, frameon=True)
axis.grid(which="major", color="0.82", linewidth=0.7)
axis.grid(which="minor", color="0.92", linewidth=0.45)

inset_axis = axis.inset_axes([0.57, 0.12, 0.40, 0.35])
for curve_key, _, color, line_style in curve_styles:
    inset_axis.plot(
        HALO_MASSES_MSUN,
        curves[curve_key],
        color=color,
        linestyle=line_style,
        linewidth=1.4,
    )

inset_axis.set_xscale("log")
inset_axis.set_yscale("log")
inset_axis.set_xlim(1.0e3, 1.0e6)
inset_axis.set_ylim(1.0e-15, 2.0e-13)
inset_axis.tick_params(axis="both", which="both", labelsize=7)
inset_axis.grid(which="major", color="0.84", linewidth=0.55)
inset_axis.grid(which="minor", color="0.93", linewidth=0.35)

figure.tight_layout()
figure_path = PROJECT_ROOT / "fig3_reproduction.png"
figure.savefig(figure_path, dpi=240, bbox_inches="tight")
print(f"图片已保存：{figure_path}")
plt.show()

图片已保存：D:\pbh formation\pbh merge\2body channel and numerical simulation\fig3_reproduction.png


C:\Users\admin\AppData\Local\Temp\ipykernel_18760\1803557683.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. 导出绘图数据

CSV 第一列是晕质量，后四列依次对应图例中的四条曲线。这样以后调整图片样式时，不必先重新计算 Appendix A(14) 积分。

In [ ]:
csv_path = PROJECT_ROOT / "fig3_reproduction.csv"
figure3_table = np.column_stack([
    HALO_MASSES_MSUN,
    curves["ludlow16_monochromatic"],
    curves["prada12_monochromatic"],
    curves["ludlow16_lognormal"],
    curves["prada12_lognormal"],
])

np.savetxt(
    csv_path,
    figure3_table,
    delimiter=",",
    header=(
        "halo_mass_msun,"
        "ludlow16_monochromatic_per_year,"
        "prada12_monochromatic_per_year,"
        "ludlow16_lognormal_per_year,"
        "prada12_lognormal_per_year"
    ),
    comments="",
    fmt="%.10e",
)

print(f"数据已保存：{csv_path}")